# Redes Neuras Convolucionais (CNNs)

### Disciplina de Sistemas Inteligentes

### Departamento de Informática e Estatística

### Profa. Nathalia da Cruz Alves


**Nota 1**: Em caso de reprodução, publicação ou compartilhamento deste material, a autoria deverá ser obrigatoriamente mantida.

**Nota 2**: Este material foi elaborado com o auxílio de ferramentas de Inteligência Artificial (IA), utilizadas para o refinamento de explicações e revisão de trechos de código. A autoria e a responsabilidade pelo conteúdo final permanecem com a autora.

Estudante(s): insira seu nome aqui


**Observação**: Solicita-se que sejam informados apenas os nomes dos estudantes efetivamente presentes e participantes da atividade.

Nesta aula você vai aprender, na prática, como construir e treinar Redes Neurais Convolucionais (CNNs) para classificação de imagens.

Como objetivos, você verá:
- Preparação de imagens para CNNs
- Construção de camadas convolucionais
- Pooling
- Extração automática de características
- Aumento de dados

Antes de rodar o treinamento dos modelos, verifiqum se o ambiente de execução está configurado para usar GPU.

- Vá em Ambiente de execução (Runtime) -> Alterar tipo de ambiente de execução
- Em Acelerador de hardware, selecione GPU
- Clique em Salvar

### Configurações iniciais

Import de bibliotecas

In [ ]:
# Construção e treinamento das redes neurais
import tensorflow as tf
from tensorflow import keras

# Visualização gráfica
import matplotlib.pyplot as plt
import seaborn as sns

# Controle de aleatoriedade
import random

# Manipulação de dados
import numpy as np
import pandas as pd

Controle de reprodutibilidade

In [ ]:
# Set da seed das bibliotecas para reprodução de resultados
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

Confirmar que a GPU está ativa

- Se aparecer algo como **PhysicalDevice('GPU:0')**, está tudo certo.

In [ ]:
tf.config.list_physical_devices('GPU')

### Conjunto de dados: Dataset MNIST

O MNIST é um dos datasets mais famosos da área de redes neurais

O dataset MNIST contém:

- 70.000 imagens de dígitos manuscritos
- imagens em escala de cinza
- tamanho 28x28 pixels
- 10 classes (números de 0 a 9 manuscritos)

In [ ]:
# Carregando o dataset MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

O MNIST já vem dividido em:

- 60.000 imagens para treinamento
- 10.000 imagens para teste

O conjunto de treino é utilizado para aprender os padrões.

O conjunto de teste é utilizado para verificar a capacidade de generalização do modelo.

In [ ]:
print(X_train.shape)  # (60000, 28, 28)
print(X_test.shape)   # (10000, 28, 28)

#### Visualização das imagens

Antes de treinar os modelos, é fundamental explorar e visualizar as imagens para:

- Identificar o tipo de imagem e suas principais características
- Avaliar a qualidade, ruídos e possíveis inconsistências
- Reconhecer formas, texturas, cores e características relevantes para o modelo
- Detectar problemas no conjunto de dados

In [ ]:
# @title Visualizando 25 imagens aleatórias do conjunto de dados

plt.figure(figsize=(10, 10))

for i in range(25):
    # Escolhe índice aleatório
    idx = np.random.randint(len(X_train))

    plt.subplot(5, 5, i + 1)

    # Exibe imagem
    plt.imshow(X_train[idx], cmap='gray')

    # Exibe rótulo correto
    plt.title(y_train[idx])

    # Remove eixos
    plt.axis('off')

plt.tight_layout()
plt.show()

#### Distribuição das classes

É importante verificar se todas as classes aparecem em quantidades semelhantes.

Datasets desbalanceados podem fazer com que o modelo favoreça as classes mais frequentes e tenha dificuldade em reconhecer as menos representadas.

In [ ]:
# @title Visulizando a distribuição das classes
classes, counts = np.unique(y_train, return_counts=True)

plt.bar(classes, counts)
plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

#### Normalização dos dados

Redes neurais aprendem melhor quando:

- entradas estão na mesma escala
- números são pequenos
- distribuição é estável


As imagens do Keras MNIST vêm com valores de pixels entre 0 a 255
- 0 = preto
- 255 = branco

Dividindo por 255 os valores ficam entre 0.0 a 1.0

Isso melhora a estabilidade do gradiente e a velocidade de treinamento

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0

### Preparação para CNN

As CNNs esperam imagens no formato:
- (amostras, altura, largura, canais)

In [ ]:
X_train = X_train.reshape(-1,28,28,1) # O argumento -1 calcula automaticamente o número de imagens.
X_test = X_test.reshape(-1,28,28,1)

print(X_train.shape)

### Primeira CNN

Objetivo: criar uma primeira CNN simples para servir como referência inicial.

Arquitetura:
- A camada de entrada recebe imagens de 28 × 28 pixels com 1 canal (escala de cinza)
- A primeira camada é uma camada convolucional (Conv2D) com 4 filtros, kernel 3×3 e ativação ReLU para extrair características das imagens
- A segunda camada é de pooling (MaxPooling2D) com janela 2×2 para redução da dimensionalidade espacial
- A terceira camada é uma camada de achatamento (Flatten) que transforma os mapas de características em um vetor unidimensional
- A camada oculta da MLP é totalmente conectada (Dense) com 16 neurônios e ativação ReLU para aprender padrões a partir das características extraídas
- A camada de saída (Dense) contém 10 neurônios com ativação Softmax, correspondentes às 10 classes do MNIST

In [ ]:
model1 = tf.keras.Sequential([

    # Camada de entrada
    tf.keras.Input(shape=(28, 28, 1)),

    # Camada convolucional
    tf.keras.layers.Conv2D(
        filters=4,
        kernel_size=(3,3),
        activation='relu',
    ),

    # Camada de pooling
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    # Camada de achatamento
    tf.keras.layers.Flatten(),

    # Camada oculta com 16 neurônios
    tf.keras.layers.Dense(
        16,
        activation='relu'
    ),

    # Camada de saída com 10 classes
    tf.keras.layers.Dense(
        10,
        activation='softmax'
    )
])

Visualização da arquitetura

In [ ]:
model1.summary()

Perguntas:

- Quantos filtros existem?

- Quantos parâmetros foram criados?

- O que cada filtro aprende?

Compilação do modelo

- **Otimizador (Adam)** algoritmo de otimização baseado na descida em gradiente, mas utiliza adaptações para ajustar a taxa de aprendizado de cada parâmetro individualmente

- **Função de perda (sparse_categorical_crossentropy)** utilizada na classificação multiclasse quando os rótulos são inteiros, como os dígitos de 0 a 9

- **Métrica de acurácia (accuracy)** mede a proporção de previsões corretas do modelo




In [ ]:
model1.compile(

    # Otimizador adaptativo
    optimizer='adam',

    # Função de perda para classificação multiclasse
    # A sparse_categorical_crossentropy é usada quando:
    # - temos múltiplas classes
    # - os rótulos são inteiros (0 até 9)
    loss='sparse_categorical_crossentropy',

    # Métrica de avaliação
    metrics=['accuracy']
)

#### Treinamento

- **Acurácia (Accuracy)** proporção de previsões corretas do modelo
- **Perda (Loss)** mede o erro das previsões; quanto menor, melhor

Durante o treinamento observe:
- **accuracy**: acurácia no conjunto de treino
- **loss**: perda no conjunto de treino
- **val_accuracy**: acurácia no conjunto de validação
- **val_loss**: perda no conjunto de validação



In [ ]:
history1 = model1.fit(
    # Usamos apenas os 60.000 dados do conjunto de treinamento
    X_train,
    y_train,

    # Número de épocas
    epochs=10,

    # 20% dos dados de treino serão usados para validação
    validation_split=0.2
)

#### Visualização dos resultados do treinamento

- **Gráfico de acurácia (accuracy)**: representa como a capacidade de classificação da rede evolui ao longo das épocas, comparando o desempenho nos dados de treino e validação

- **Gráfico de perda (loss)**: representa como o erro das previsões evolui durante o treinamento, também comparando treino e validação





In [ ]:
# @title Visualizando gráficos de acurácia e perda no treinamento

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de acurácia
axes[0].plot(history1.history['accuracy'])
axes[0].plot(history1.history['val_accuracy'])
axes[0].set_title('CNN - Acurácia')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Acurácia')
axes[0].legend(['Treino', 'Validação'])
axes[0].grid(True)

# Gráfico de perda
axes[1].plot(history1.history['loss'])
axes[1].plot(history1.history['val_loss'])
axes[1].set_title('CNN - Perda')
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Perda')
axes[1].legend(['Treino', 'Validação'])
axes[1].grid(True)

plt.tight_layout()
plt.show()

Visualização dos filtros

Como CNNs são frequentemente consideradas "caixas-pretas", visualizar seus filtros ajuda a interpretar quais padrões cada camada detecta.

In [ ]:
# @title Visualizando filtros da camada convolucional do modelo

filters, biases = model1.layers[0].get_weights()
print("Forma dos filtros:", filters.shape)

num_filters = filters.shape[3]

plt.figure(figsize=(10, 5))

for i in range(num_filters):
    plt.subplot(2, 4, i + 1)
    plt.imshow(filters[:, :, 0, i], cmap='gray')
    plt.title(f"Filtro {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()

### CNN Mais Profunda

Objetivo: criar uma CNN com mais camadas convolucionais para observar o efeito de profundidade e uso de mais filtros.

Arquitetura:
- A camada de entrada recebe imagens de 28 × 28 pixels com 1 canal (escala de cinza)
- A primeira camada convolucional (Conv2D) com 8 filtros, kernel 3×3 e função de ativação ReLU
- A segunda camada convolucional (Conv2D) com 16 filtros, kernel 3×3 e função de ativação ReLU
- A terceira camada convolucional (Conv2D) com 32 filtros, kernel 3×3 e função de ativação ReLU
- A quarta camada é de MaxPooling2D com janela 2×2 para redução da dimensionalidade espacial
- A quinta camada é uma camada Flatten que transforma os mapas de características em um vetor
- A camada oculta da MLP é totalmente conectada (Dense) com 64 neurônios e função de ativação ReLU
- A camada de saída contém 10 neurônios com ativação Softmax, correspondentes às 10 classes do MNIST

Observe, ao longo das épocas, o comportamento da acurácia (accuracy) e da função de perda (loss) nos conjuntos de treinamento e validação. Verifique se o modelo está aprendendo adequadamente e se há sinais de sobreajuste (overfitting) ou subajuste (underfitting).


In [ ]:
model2 = tf.keras.Sequential([

    # Camada de entrada
    tf.keras.Input(shape=(28, 28, 1)),

    # Primeira camada convolucional
    tf.keras.layers.Conv2D(
        filters=8,
        kernel_size=(3,3),
        activation='relu'
    ),

    # Segunda camada convolucional
    tf.keras.layers.Conv2D(
        filters=16,
        kernel_size=(3,3),
        activation='relu'
    ),

    # Terceira camada convolucional
    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation='relu'
    ),

    # Camada de pooling
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    # Camada de achatamento
    tf.keras.layers.Flatten(),

    # Camada oculta
    tf.keras.layers.Dense(
        64,
        activation='relu'
    ),

    # Camada de saída
    tf.keras.layers.Dense(
        10,
        activation='softmax'
    )
])

model2.compile(

    # Otimizador adaptativo
    optimizer='adam',

    # Função de perda para classificação multiclasse
    # A sparse_categorical_crossentropy é usada quando:
    # - temos múltiplas classes
    # - os rótulos são inteiros (0 até 9)
    loss='sparse_categorical_crossentropy',

    # Métrica de avaliação
    metrics=['accuracy']
)

history2 = model2.fit(
    # Usamos apenas os 60.000 dados do conjunto de treinamento
    X_train,
    y_train,

    # Número de épocas
    epochs=10,

    # 20% dos dados de treino serão usados para validação
    validation_split=0.2
)

#### Visualização dos resultados do treinamento

Plote o gráfico de visualização dos resultados de treinamento e responda:
- Qual o padrão observado a cada época com a acurácia e a perda no conjunto de treinamento e de validação?
- A CNN Profunda apresentou melhor desempenho do que o primeiro modelo?


**Overfitting vs. Underfitting**

- **Underfitting (subajuste)**: ocorre quando o modelo é demasiado simples ou não treinou tempo suficiente para conseguir capturar os padrões dos dados.
- - Sintomas nos gráficos: a acurácia é baixa e a perda (loss) é alta tanto no treino como na validação.

- **Overfitting (sobreajuste)**: ocorre quando o modelo é demasiado complexo e acaba por "decorar" os dados de treino em vez de aprender os conceitos gerais.
- - Sintomas nos gráficos: a linha de treino continua a melhorar (perda a descer, acurácia a subir até ao topo), mas a linha de validação estanca ou começa a piorar (a perda sobe e a acurácia desce).

In [ ]:
# @title Visualizando gráficos de acurácia e perda no treinamento

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de acurácia
axes[0].plot(history2.history['accuracy'])
axes[0].plot(history2.history['val_accuracy'])
axes[0].set_title('CNN Profunda - Acurácia')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Acurácia')
axes[0].legend(['Treino', 'Validação'])
axes[0].grid(True)

# Gráfico de perda
axes[1].plot(history2.history['loss'])
axes[1].plot(history2.history['val_loss'])
axes[1].set_title('CNN Profunda - Perda')
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Perda')
axes[1].legend(['Treino', 'Validação'])
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# @title Visualizando filtros da primeira camada do modelo

filters, biases = model2.layers[0].get_weights()
print("Forma dos filtros:", filters.shape)

num_filters = filters.shape[3]

plt.figure(figsize=(10, 5))

for i in range(num_filters):
    plt.subplot(2, 4, i + 1)
    plt.imshow(filters[:, :, 0, i], cmap='gray')
    plt.title(f"Filtro {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()

### Experimentação com CNNs

Crie e treine 3 modelos com diferentes características, isto é, modifique a quantidade de camadas convolucionais e quantidade de filtros.
Use diferentes quantidade de épocas nos experimentos.


Responda:
- Algum modelo teve spbreajuste (overfitting)? Justifique.

- Algum modelo teve subajuste (underfitting)? Justifique.

- Qual modelo performou melhor? Justifique.

### CNN com aumento de dados

Para aumentar a variedade dos dados de treinamento e melhorar a capacidade de generalização do modelo, a etapa de Data Augmentation é útil.

No exemplo a seguir, ela é composta pelas seguintes transformações aleatórias:

- RandomRotation(0.1): realiza pequenas rotações aleatórias nas imagens, simulando diferentes orientações dos dígitos.
- RandomTranslation(0.1, 0.1): desloca as imagens horizontal e verticalmente em até 10% do seu tamanho, tornando o modelo mais robusto a mudanças de posição.
- RandomZoom(0.1): aplica pequenos aumentos ou reduções no tamanho da imagem, ajudando o modelo a lidar com variações de escala.

Essas transformações são aplicadas apenas durante o treinamento, gerando novas versões das imagens a cada época sem alterar o conjunto de dados original. Isso ajuda a reduzir o overfitting e melhora a capacidade do modelo de reconhecer padrões em dados não vistos anteriormente.

In [ ]:
data_augmentation = keras.Sequential([

    keras.layers.RandomRotation(0.1),

    keras.layers.RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    ),

    keras.layers.RandomZoom(0.1)
])

In [ ]:
# @title Visualizando o aumento de dados
plt.figure(figsize=(10,10))

img = X_train[0]

for i in range(9):

    augmented = data_augmentation(
        tf.expand_dims(img, 0)
    )

    plt.subplot(3,3,i+1)

    plt.imshow(
        augmented[0].numpy().squeeze(),
        cmap='gray'
    )

    plt.axis('off')

plt.show()

Treine o modelo a seguir com aumento de dados

In [ ]:
model3 = keras.Sequential([

    keras.layers.Input(shape=(28,28,1)),

    data_augmentation,

    tf.keras.layers.Conv2D(
        filters=8,
        kernel_size=(3,3),
        activation='relu',
    ),

    tf.keras.layers.Conv2D(
        filters=16,
        kernel_size=(3,3),
        activation='relu'
    ),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation='relu'
    ),

    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        64,
        activation='relu'
    ),

    tf.keras.layers.Dense(
        10,
        activation='softmax'
    )
])

# Compilação
model3.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Treinamento
history3 = model3.fit(
    X_train,
    y_train,
    epochs=10,
    validation_split=0.2
)

#### Visualização dos resultados do treinamento

Plote o gráfico de visualização dos resultados de treinamento e responda:
- Qual o padrão observado a cada época com a acurácia e a perda no conjunto de treinamento e de validação?
- A CNN com aumento de dados apresentou melhor desempenho?

In [ ]:
# @title Visualizando gráficos de acurácia e perda no treinamento

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de Acurácia
axes[0].plot(history3.history['accuracy'])
axes[0].plot(history3.history['val_accuracy'])
axes[0].set_title('CNN Aumento de dados - Acurácia')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Acurácia')
axes[0].legend(['Treino', 'Validação'])
axes[0].grid(True)

# Gráfico de Perda
axes[1].plot(history3.history['loss'])
axes[1].plot(history3.history['val_loss'])
axes[1].set_title('CNN Aumento de dados - Perda')
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Perda')
axes[1].legend(['Treino', 'Validação'])
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Avaliação final

Utilize o conjunto de testes para avaliação do modelo.

O conjunto de teste:
- nunca deve ter sido visto durante treinamento
- simula mundo real

In [ ]:
test_loss, test_acc = model3.evaluate(
    X_test,
    y_test
)

# Probabilidades previstas
y_probs = model3.predict(X_test)

# Classe prevista
y_pred = np.argmax(y_probs, axis=1)

print(f"Perda no conjunto de teste: {test_loss:.4f}")
print(f"Acurácia no conjunto de teste: {test_acc:.4f}")

### Testando o aumento de dados

Para testar o aumento de dados, as imagens de teste também serão rotacionadas artificialmente.

In [ ]:
# @title Função para avaliar modelos com aumento de dados
def avaliar_modelos_com_augmentation(
    modelos,
    data_augmentation,
    X_test,
    y_test,
    n=10
):

    acertos = {nome: 0 for nome, _ in modelos}
    exemplos = []

    for _ in range(n):

        idx = np.random.randint(len(X_test))

        img = X_test[idx]
        real = y_test[idx]

        img_tensor = tf.expand_dims(img, axis=0)

        img_torta = data_augmentation(
            img_tensor,
            training=True
        )[0]

        img_input = np.expand_dims(
            img_torta.numpy(),
            axis=0
        )

        predicoes = {}

        for nome, modelo in modelos:

            pred = modelo.predict(
                img_input,
                verbose=0
            )

            classe = np.argmax(pred)

            predicoes[nome] = classe

            if classe == real:
                acertos[nome] += 1

        exemplos.append(
            (
                img,
                img_torta.numpy(),
                real,
                predicoes
            )
        )

    print("Resultados:")

    for nome, _ in modelos:

        acc = acertos[nome] / n

        print(
            f"{nome} - Acurácia: {acc:.4f}"
        )

    plt.figure(figsize=(2*n, 6))

    for i, (_, img_torta, real, predicoes) in enumerate(exemplos):

        # Linha 1: imagem aumentada
        plt.subplot(2, n, i + 1)

        plt.imshow(
            img_torta.squeeze(),
            cmap='gray'
        )

        plt.title(f"Real: {real}")

        plt.axis('off')

        # Linha 2: predições
        texto = "\n".join(
            [
                f"{nome}: {classe}"
                for nome, classe in predicoes.items()
            ]
        )

        plt.subplot(2, n, i + 1 + n)

        plt.imshow(
            img_torta.squeeze(),
            cmap='gray'
        )

        plt.title(
            texto,
            fontsize=8
        )

        plt.axis('off')

    plt.tight_layout()

    plt.show()

Comparação entre os modelos

In [ ]:
modelos = [
    ("M1", model1),
    ("M2", model2),
    ("M3", model3),
]

avaliar_modelos_com_augmentation(
    modelos,
    data_augmentation,
    X_test,
    y_test,
    n=60
)

Qual modelo teve mais facilidade para reconhecer as imagem com aumento de dados?